# 41 — Pruning Policies

## How should revision decisions be applied?

Notebook 37 introduced guided pruning.

Notebook 41 asks:

> What policy should govern pruning decisions once triggers, residues, and budgets are known?

Outputs:

```text
results/41_pruning_policies.csv
figures/41_pruning_policies.png
```

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git","clone",REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print(repo)

In [ ]:
pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg/"paths.py").write_text('''from pathlib import Path
def ensure_dirs(root, names=("results","figures","reports")):
    root = Path(root)
    out={}
    for n in names:
        p=root/n
        p.mkdir(parents=True, exist_ok=True)
        out[n]=p
    return out
''', encoding="utf-8")

(pkg/"policies.py").write_text('''def policy_score(residue, trigger, budget):
    return residue + budget - trigger

def policy_name(score):
    if score >= 5:
        return "preserve_and_refine"
    if score >= 2:
        return "targeted_revision"
    return "major_restructure"
''', encoding="utf-8")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

In [ ]:
rows = [
    ["preserve strong residues",5,1,2],
    ["moderate revision",3,2,2],
    ["budget constrained",2,3,1],
    ["high drift",1,4,1],
    ["restructure",0,5,0],
]

df = pd.DataFrame(
    rows,
    columns=["policy","residue_strength","trigger_pressure","budget_headroom"]
)

df["policy_score"] = (
    df["residue_strength"]
    + df["budget_headroom"]
    - df["trigger_pressure"]
)

df

In [ ]:
csv_path = results / "41_pruning_policies.csv"
df.to_csv(csv_path, index=False)
print(csv_path)

In [ ]:
fig_path = figures / "41_pruning_policies.png"

ax = df.plot.bar(
    x="policy",
    y="policy_score",
    legend=False,
    figsize=(10,5),
)

ax.set_title("Pruning policies")
ax.set_ylabel("Policy score")

plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print(fig_path)

## Result

Notebook 41 converts guidance into policy choices.

Next:

```text
43_policy_evaluation.ipynb
```